In [4]:
import tensorflow as tf
from tensorflow.keras import layers,models
import pandas as pd
import os


styles_df = pd.read_csv('../data/styles.csv', on_bad_lines='skip')

print("Columns available:", styles_df.columns)

IMAGE_DIR = '../data/images'
TARGET_COLUMN = 'masterCategory'  
ID_COLUMN = 'id'                  

styles_df['image_path'] = styles_df[ID_COLUMN].astype(str) + '.jpg'
styles_df['full_path'] = styles_df['image_path'].apply(lambda x: os.path.join(IMAGE_DIR, x))
styles_df = styles_df[styles_df['full_path'].apply(os.path.exists)]

print(f"Matched {len(styles_df)} images with metadata.")

class_names = styles_df[TARGET_COLUMN].unique()
class_to_idx = {cls_name: i for i, cls_name in enumerate(class_names)}
idx_to_class = {i: cls_name for i, cls_name in enumerate(class_names)}

image_paths = styles_df['full_path'].values
labels = styles_df[TARGET_COLUMN].map(class_to_idx).values

IMG_SIZE = (128, 128)
BATCH_SIZE = 32

def load_and_preprocess_image(path, label):
    image = tf.io.read_file(path)
    image = tf.image.decode_jpeg(image, channels=3)
    image = tf.image.resize(image, IMG_SIZE)
    image = image / 255.0 
    return image, label

val_split = int(0.2 * len(image_paths))

train_paths, val_paths = image_paths[val_split:], image_paths[:val_split]
train_labels, val_labels = labels[val_split:], labels[:val_split]

train_dataset = tf.data.Dataset.from_tensor_slices((train_paths, train_labels))
train_dataset = train_dataset.map(load_and_preprocess_image, num_parallel_calls=tf.data.AUTOTUNE)

val_dataset = tf.data.Dataset.from_tensor_slices((val_paths, val_labels))
val_dataset = val_dataset.map(load_and_preprocess_image, num_parallel_calls=tf.data.AUTOTUNE)

data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
])

AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_dataset.shuffle(1000).batch(BATCH_SIZE)
train_ds = train_ds.map(lambda x, y: (data_augmentation(x, training=True), y), num_parallel_calls=AUTOTUNE)
train_ds = train_ds.prefetch(buffer_size=AUTOTUNE)

val_ds = val_dataset.batch(BATCH_SIZE).prefetch(buffer_size=AUTOTUNE)

print("Dataset pipeline successfully created!")

Columns available: Index(['id', 'gender', 'masterCategory', 'subCategory', 'articleType',
       'baseColour', 'season', 'year', 'usage', 'productDisplayName'],
      dtype='str')
Matched 44419 images with metadata.
Dataset pipeline successfully created!


In [5]:
from tensorflow.keras.applications import VGG16
from tensorflow.keras.callbacks import EarlyStopping

base_model = VGG16(
    weights='imagenet', 
    include_top=False, 
    input_shape=(128, 128, 3)
)

base_model.trainable = False

model = models.Sequential([
    base_model,
    layers.Flatten(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(len(class_names), activation='softmax')
])


model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

early_stopping = EarlyStopping(
    monitor='val_loss',      
    patience=3,               
    restore_best_weights=True 
)


EPOCHS = 5
history_frozen = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=[early_stopping]
)

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ vgg16 (Functional)              │ (None, 4, 4, 512)      │    14,714,688 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 8192)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 256)            │     2,097,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 7)              │         1,799 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 16,813,895 (64.14 MB)

 Trainable params: 2,099,207 (8.01 MB)

 Non-trainable params: 14,714,688 (56.13 MB)

Epoch 1/5


/home/sayak/Documents/RetailMind AI/.venv/lib/python3.12/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


1111/1111 ━━━━━━━━━━━━━━━━━━━━ 922s 829ms/step - accuracy: 0.9327 - loss: 0.2173 - val_accuracy: 0.9634 - val_loss: 0.1296
Epoch 2/5
1111/1111 ━━━━━━━━━━━━━━━━━━━━ 943s 848ms/step - accuracy: 0.9559 - loss: 0.1544 - val_accuracy: 0.9770 - val_loss: 0.0865
Epoch 3/5
1111/1111 ━━━━━━━━━━━━━━━━━━━━ 955s 859ms/step - accuracy: 0.9611 - loss: 0.1367 - val_accuracy: 0.9720 - val_loss: 0.0890
Epoch 4/5
1111/1111 ━━━━━━━━━━━━━━━━━━━━ 950s 855ms/step - accuracy: 0.9633 - loss: 0.1260 - val_accuracy: 0.9780 - val_loss: 0.0779
Epoch 5/5
1111/1111 ━━━━━━━━━━━━━━━━━━━━ 953s 858ms/step - accuracy: 0.9661 - loss: 0.1193 - val_accuracy: 0.9815 - val_loss: 0.0718


In [9]:

base_model.trainable = True

for layer in base_model.layers[:15]:
    layer.trainable = False


from tensorflow.keras.optimizers import Adam

model.compile(
    optimizer=Adam(learning_rate=1e-5), 
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)


FINE_TUNE_EPOCHS = 3
total_epochs = EPOCHS + FINE_TUNE_EPOCHS

history_fine = model.fit(
    train_ds,
    validation_data=val_ds,
    initial_epoch=history_frozen.epoch[-1] + 1,
    epochs=total_epochs
)

val_loss, val_acc = model.evaluate(val_ds)
print(f"\nFine-Tuned Validation Accuracy: {val_acc * 100:.2f}%")

Epoch 6/8


1111/1111 ━━━━━━━━━━━━━━━━━━━━ 1206s 1s/step - accuracy: 0.9752 - loss: 0.0902 - val_accuracy: 0.9844 - val_loss: 0.0664
Epoch 7/8
1111/1111 ━━━━━━━━━━━━━━━━━━━━ 1213s 1s/step - accuracy: 0.9813 - loss: 0.0732 - val_accuracy: 0.9854 - val_loss: 0.0634
Epoch 8/8
1111/1111 ━━━━━━━━━━━━━━━━━━━━ 1220s 1s/step - accuracy: 0.9845 - loss: 0.0612 - val_accuracy: 0.9889 - val_loss: 0.0529
278/278 ━━━━━━━━━━━━━━━━━━━━ 191s 687ms/step - accuracy: 0.9889 - loss: 0.0529

Fine-Tuned Validation Accuracy: 98.89%


In [10]:
import joblib 
model_path = '../models/classifier_vgg.joblib'
joblib.dump(model, model_path)

print(f"Model successfully dumped using joblib to '{model_path}'!")

Model successfully dumped using joblib to '../models/classifier_vgg.joblib'!


In [16]:
import numpy as np

loaded_model = joblib.load( '../models/classifier_vgg.joblib')
print("Model successfully loaded using joblib!")

for sample_images, sample_labels in val_ds.take(1):
    single_image = sample_images[3:4]
    true_label_idx = sample_labels[0].numpy()
    
    # Run prediction
    predictions = loaded_model.predict(single_image)
    predicted_class_idx = np.argmax(predictions[0])
    confidence = np.max(predictions[0]) * 100
    
    # Print results
    print(f"\n--- Prediction Result ---")
    print(f"True Category: {class_names[true_label_idx]}")
    print(f"Predicted Category: {class_names[predicted_class_idx]}")
    print(f"Confidence: {confidence:.2f}%")
    break

Model successfully loaded using joblib!
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 152ms/step

--- Prediction Result ---
True Category: Apparel
Predicted Category: Apparel
Confidence: 100.00%
